In [27]:
%load_ext autoreload
%autoreload 2

from utils_gae import *
from models_encoders import *

import numpy as np
import pandas as pd
import io
import html
import networkx as nx
import os
import sys

path_to_src = os.path.abspath(os.path.join('..'))
if path_to_src not in sys.path:
    sys.path.append(path_to_src)
from SHAP_like_graph_tool import load_all_data_for_graph, loadsave_data_joblib, hide_graph_links

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
def load_graphml_safe(path):
        with open(path, 'r', encoding='utf-8') as f:
            raw_data = f.read()

        clean_data = html.unescape(raw_data)
        G = nx.read_graphml(io.StringIO(clean_data))
        
        print(f"✅ Graphe chargé : {G.number_of_nodes()} nœuds et {G.number_of_edges()} liens.")
        return G

In [34]:
for i in np.arange(0.10, 0.20, 0.10):
    sbm_val = f"{i:.2f}"
    pos_val = f"{1-i:.2f}"
    G_name = f"artificial_graph_sbmv4_{sbm_val.replace('.', '_')}_pos_{pos_val.replace('.', '_')}"
    G_name_obj = f"artificial_graph_sbmv4_{sbm_val.replace('.', '_')}_pos_{pos_val.replace('.', '_')}"

    G_kept = loadsave_data_joblib(filename=f"G_kept_w_struct_com_dist_{G_name}", mode="load")
    G_train, G_test = hide_graph_links(G_kept, test_size=0.15)
    
    # 1. Chargement des données d'entraînement
    G_train, dataset_train, dataset_eval, _, _, _ = load_all_data_for_graph(G_name)

    print(f"--- Inspection de {G_name} ---")
    
    # Affiche les attributs du premier nœud pour trouver la clé de position (ex: 'pos', 'coords', 'x'...)
    if len(G_train.nodes) > 0:
        first_node = list(G_train.nodes(data=True))[0]
        print(f"Attributs du premier nœud ({first_node[0]}) :", first_node[1])
    print("-" * 30)
    # ---------------------------
    
    kakak = prepare_graph_data(G_train, None, "GT_pos")
    kakak_eval = prepare_graph_data(G_test, None, "GT_pos")
    

Graphe original: 2406 liens
Graphe d'entraînement: 2045 liens
Liens cachés pour le test: 361
SHAP Analysis introuvable pour artificial_graph_sbmv4_0_10_pos_0_90.
--- Inspection de artificial_graph_sbmv4_0_10_pos_0_90 ---
Attributs du premier nœud (0) : {'GT_pos': array([0.92292958, 0.70103991, 0.73655641, 0.17676583]), 'degree': 11, 'pr': 0.003038510336948907, 'ppr': 0.00389401636703961, 'lcc': 0.3090909090909091, 'and': 22.545454545454547, 'dc': 0.055837563451776644, 'katz': 0.03391233260865749, 'n2v_homophily': array([-0.2217587 , -0.35322562,  0.1736436 ,  0.5068727 ,  0.3025488 ,
        0.2693962 ,  0.29632103, -0.29388478, -0.38745776,  0.1106621 ,
        0.2123055 ,  0.17527723, -0.36340067, -0.1566838 ,  0.40776983,
        0.57656085, -0.1844689 , -0.11100913,  0.11211573,  0.4153304 ,
        0.13983795,  0.13951547, -0.07095178, -0.22800872,  0.25835326,
        0.17733885, -0.11540537,  0.09557918,  0.00575741, -0.16663735,
        0.1422409 ,  0.0391185 , -0.05138953,  0.

In [38]:
print(kakak)
print(kakak_eval)

gae = GAE(in_channels=1, out_channels=16)
gae_disentangled = GAE(in_channels=1, out_channels=16)
gae_bis = GAE(in_channels=1, out_channels=16)
gae_disentangled_bis = GAE(in_channels=1, out_channels=16)

geo = GeoEncoder(in_pos_dim=4, out_channels=16, scale=1.0)
geo_disentangled = GeoEncoder(in_pos_dim=4, out_channels=16, scale=1.0)

Joint = JointLinkPredictionModel(gae, gae_bis)
Joint_disentangled = JointLinkPredictionModel(gae_disentangled, gae_disentangled_bis)

Joint_disentangled.fit(kakak, kakak_eval, epochs=1001, lambda_ortho=0.01, patience=5000)
Joint.fit(kakak, kakak_eval, epochs=1001, disentangle=False, patience=5000)


def check_collapse(z_a, z_b):
    cos_sim = F.cosine_similarity(z_a, z_b).mean()
    print(f"Similarité Cosine moyenne entre espaces : {cos_sim.item():.4f}")

print("------DISENTANGLED----------")
z_a = Joint_disentangled.encoder_a(kakak.x, kakak.edge_index)
z_b = Joint_disentangled.encoder_b(kakak.pos)
check_collapse(z_a, z_b)
auc, ap = Joint_disentangled.evaluate(kakak_eval)
print(f"Scores sur eval set : {auc, ap}")
print("------CLASSIC----------")
z_a = Joint.encoder_a(kakak.x, kakak.edge_index)
z_b = Joint.encoder_b(kakak.pos)
check_collapse(z_a, z_b)
auc, ap = Joint.evaluate(kakak_eval)
print(f"Scores sur eval set : {auc, ap}")

Data(x=[198, 1], edge_index=[2, 4090], pos=[198, 4], num_nodes=198)
Data(x=[198, 1], edge_index=[2, 722], pos=[198, 4], num_nodes=198)
Démarrage de l'entraînement : DISENTANGLED
Modèle sauvegardé dans : ../../outputs/models/joint_disentangled_20260513.pt

[DISENTANGLED] Ep 000 | Loss: 2.6402 | Rec: 0.6602 | A: 0.641 | B: 0.642 | Lambda*Ortho: 1.9800 | VAL AUC: 0.6873 (★ Best)
[DISENTANGLED] Ep 010 | Loss: 2.3935 | Rec: 0.4257 | A: 0.632 | B: 0.631 | Lambda*Ortho: 1.9678 | VAL AUC: 0.6505
[DISENTANGLED] Ep 020 | Loss: 1.8972 | Rec: 0.3552 | A: 0.642 | B: 0.627 | Lambda*Ortho: 1.5420 | VAL AUC: 0.5737
[DISENTANGLED] Ep 030 | Loss: 1.1064 | Rec: 0.3310 | A: 0.630 | B: 0.593 | Lambda*Ortho: 0.7754 | VAL AUC: 0.5110
[DISENTANGLED] Ep 040 | Loss: 1.1896 | Rec: 0.3232 | A: 0.624 | B: 0.651 | Lambda*Ortho: 0.8664 | VAL AUC: 0.5871
[DISENTANGLED] Ep 050 | Loss: 1.1147 | Rec: 0.3259 | A: 0.622 | B: 0.618 | Lambda*Ortho: 0.7887 | VAL AUC: 0.5781
[DISENTANGLED] Ep 060 | Loss: 0.9553 | Rec: 0.3257 

TypeError: GAE.forward() missing 1 required positional argument: 'edge_index'